# BinaryGap

[Codility task](https://app.codility.com/programmers/trainings/9/binary_gap/) - Exercise 9, Bitwise operations - *easy*

A **binary gap** within a positive integer `N` is any maximal sequence of consecutive zeros that is surrounded by ones at both ends in the binary representation of `N`.

Write `solution(N)` returning the length of the longest binary gap (`0` if there is none).

| N | binary | result |
|---|--------|--------|
| 9 | `1001` | 2 |
| 529 | `1000010001` | 4 |
| 1041 | `10000010001` | 5 |
| 32 | `100000` | 0 |

Assumption: `N` is an integer in `[1..2,147,483,647]`.

## Idea

1. Write `N` in binary.
2. Trailing zeros are **not** a gap (no `1` on their right), so strip them.
3. What remains starts and ends with `1`. Splitting on `'1'` leaves the runs of zeros between ones; the answer is the longest one.

Time O(log N), because `N` has at most 31 bits.

In [ ]:
def solution(N):
    """Longest binary gap of N - string version."""
    inner = bin(N)[2:].rstrip("0")           # drop the '0b' prefix and trailing zeros
    return max((len(run) for run in inner.split("1")), default=0)

## Alternative: scan the bits

Same result in O(1) extra memory: shift `N` right and count the zeros seen since the last `1`.

In [ ]:
def solution_bits(N):
    """Longest binary gap of N - bit scan, no string conversion."""
    best = current = 0
    seen_one = False
    while N:
        if N & 1:
            if seen_one:
                best = max(best, current)
            seen_one = True
            current = 0
        else:
            current += 1
        N >>= 1
    return best

## Official examples

In [ ]:
for f in (solution, solution_bits):
    assert f(9) == 2
    assert f(529) == 4
    assert f(20) == 1
    assert f(15) == 0
    assert f(32) == 0
    assert f(1041) == 5
print("examples OK")

## Edge cases

In [ ]:
for f in (solution, solution_bits):
    assert f(1) == 0                              # '1'
    assert f(2) == 0                              # '10': trailing zero only
    assert f(3) == 0                              # '11'
    assert f(5) == 1                              # '101'
    assert f(6) == 0                              # '110'
    assert f(2**30) == 0                          # a single 1 followed by zeros
    assert f(2**30 + 1) == 29                     # '1' + 29 zeros + '1'
    assert f(2_147_483_647) == 0                  # 31 ones
    assert f(2_147_483_646) == 0                  # 30 ones and a trailing zero
print("edge cases OK")

## Brute-force check

A deliberately naive reference: use a regular expression to find every `1 0* 1` block with no `1` inside, and compare both solutions on a large range.

In [ ]:
import re


def reference(N):
    gaps = re.findall(r"(?<=1)0+(?=1)", bin(N)[2:])
    return max(map(len, gaps), default=0)


for n in range(1, 100_001):
    expected = reference(n)
    assert solution(n) == expected, n
    assert solution_bits(n) == expected, n
print("matches the reference for N = 1..100000")

## Performance

In [ ]:
import timeit

N = 2_147_483_647 - 2**20
print("string version:", timeit.timeit(lambda: solution(N), number=100_000), "s / 100k calls")
print("bits version:  ", timeit.timeit(lambda: solution_bits(N), number=100_000), "s / 100k calls")

## Notes

- `bin(N)` already has no leading zeros, so there is no leading-gap problem.
- The only subtle point is the **trailing** zeros: `32 -> 0` is the classic trap.
- Both versions are O(log N) time; the bit scan uses O(1) extra space, the string version O(log N).